In [0]:
%sql
-----  Creating new catalog, schema  ------ 
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


##### Question1: Business city table has data from the day udaan has started operation. 
Write a SQL to identify year-wise count of new cities where udaan started their operations.

In [0]:
%sql
DROP TABLE IF EXISTS business_city;
CREATE TABLE business_city (
    business_date DATE,
    city_id INT
);
INSERT INTO business_city
VALUES
('2020-01-02', 3),
('2020-07-01', 7),
('2021-01-01', 3),
('2021-02-03', 19),
('2022-12-01', 3),
('2022-12-15', 3),
('2022-02-28', 12);

SELECT * FROM business_city ORDER BY business_date;

business_date,city_id
2020-01-02,3
2020-07-01,7
2021-01-01,3
2021-02-03,19
2022-02-28,12
2022-12-01,3
2022-12-15,3


In [0]:
%sql
-- with cte as (
-- select 
--     *,
--     year(business_date) as year,
--     dense_rank() over(partition by city_id order by business_date) as rank
-- from business_city
-- )
-- select 
--     year,
--     count(city_id)
-- from cte 
-- where rank = 1
-- group by year;

with cte as (
    select
        distinct city_id,
        min(business_date) as date
    from business_city
    group by city_id
)
select
    year(date),
    count(city_id) as count
from cte
group by year(date);

year(date),count
2020,2
2021,1
2022,1


##### Question2: There are 3 rows in a movie hall each with 10 seats in each row. write a sql query to find 4 consecutive empty seats.

In [0]:
%sql
DROP TABLE IF EXISTS movie;
CREATE TABLE movie (seat VARCHAR(50), occupancy INT);
INSERT INTO movie VALUES
('a1',1), ('a2',1), ('a3',0), ('a4',0), ('a5',0), ('a6',0), ('a7',1), ('a8',1), ('a9',0), ('a10',0), ('b1',0), ('b2',0), ('b3',0), ('b4',1), ('b5',1), ('b6',1), ('b7',1), ('b8',0), ('b9',0), ('b10',0), ('c1',0), ('c2',1), ('c3',0), ('c4',1), ('c5',1), ('c6',0), ('c7',1), ('c8',0), ('c9',0), ('c10',1);

SELECT * FROM movie;

seat,occupancy
a1,1
a2,1
a3,0
a4,0
a5,0
a6,0
a7,1
a8,1
a9,0
a10,0


In [0]:
%sql
with cte as (
    select
        seat,
        left(seat,1) as row,
        substr(seat,2) as seat_no,
        occupancy
    from movie
)
select *,
    case
        when (seat_no - occupancy) = seat_no then "empty"
        else "occupied"
    end as status
from cte;

seat,row,seat_no,occupancy,status
a1,a,1,1,occupied
a2,a,2,1,occupied
a3,a,3,0,empty
a4,a,4,0,empty
a5,a,5,0,empty
a6,a,6,0,empty
a7,a,7,1,occupied
a8,a,8,1,occupied
a9,a,9,0,empty
a10,a,10,0,empty


##### Question 4: determine phone numbers that satisfy below conditions: 
> - the numbers have both incoming & outgoing calls
> - the sum of duration of outgoing calls should be greater than sum of duration of incoming calls

In [0]:
%sql
DROP TABLE IF EXISTS call_details;
CREATE TABLE call_details (
    call_type VARCHAR(10),
    call_number VARCHAR(12),
    call_duration INT
);
INSERT INTO call_details VALUES ('OUT','181868',13), ('OUT','2159010',8), ('OUT','2159010',178), ('SMS','4153810',1), ('OUT','2159010',152), ('OUT','9140152',18), ('SMS','4162672',1), ('SMS','9168204',1), ('OUT','9168204',576), ('INC','2159010',5), ('INC','2159010',4), ('SMS','2159010',1), ('SMS','4535614',1), ('OUT','181868',20), ('INC','181868',54), ('INC','218748',20), ('INC','2159010',9), ('INC','197432',66), ('SMS','2159010',1), ('SMS','4535614',1);

SELECT * FROM call_details;

call_type,call_number,call_duration
OUT,181868,13
OUT,2159010,8
OUT,2159010,178
SMS,4153810,1
OUT,2159010,152
OUT,9140152,18
SMS,4162672,1
SMS,9168204,1
OUT,9168204,576
INC,2159010,5


In [0]:
%sql
with cte as (
    select
        call_number,
        sum(case when call_type = 'OUT' then call_duration else null end) as out_duration,
        sum(case when call_type = 'INC' then call_duration else null end) as inc_duration
    from call_details
    group by call_number
)
select * from cte
where out_duration is not null and inc_duration is not null and out_duration > inc_duration;

call_number,out_duration,inc_duration
2159010,338,18


##### Question5: FInd in which innings & match, player covered his milestones of (1000, 5000, 10000, 15000) & make it in dynamic way.

In [0]:
%sql
DROP TABLE IF EXISTS sachin_batting_scores;

CREATE TABLE sachin_batting_scores (
    Match INT,
    Innings INT,
    match_date DATE,
    Versus VARCHAR(100),
    Ground VARCHAR(200),
    How_Dismissed VARCHAR(500),
    Runs INT,
    Balls_faced INT,
    strike_rate DOUBLE
);

INSERT INTO sachin_batting_scores
    (Match, Innings, match_date, Versus, Ground, How_Dismissed, Runs, Balls_faced, strike_rate)
VALUES
(1, 1, '1989-12-18', 'Pakistan', 'Jinnah Stadium (Gujwranwala)', 'c Wasim Akram b Waqar Younis', 0, 2, '0.0'),
(2, 2, '1990-03-01', 'New Zealand', 'Carisbrook', 'c & b S A Thomson', 0, 2, '0.0'),
(3, 3, '1990-03-06', 'New Zealand', 'Basin Reserve', 'c †I D S Smith b S A Thomson', 36, 39, '92.31'),
(4, 4, '1990-04-25', 'Sri Lanka', 'Sharjah Cricket Stadium', 'run out', 10, 12, '83.33'),
(5, 5, '1990-04-27', 'Pakistan', 'Sharjah Cricket Stadium', 'c Saeed Anwar b Imran Khan', 20, 25, '80.0'),
(6, 6, '1990-07-18', 'England', 'Headingley', 'b D E Malcolm', 19, 35, '54.29'),
(7, 7, '1990-07-20', 'England', 'Trent Bridge', 'b A R C Fraser', 31, 26, '119.23'),
(8, 8, '1990-12-01', 'Sri Lanka', 'Vidarbha Cricket Association Ground', 'b R J Ratnayake', 36, 22, '163.64'),
(9, 9, '1990-12-05', 'Sri Lanka', 'Nehru Stadium (Pune)', 'b G F Labrooy', 53, 41, '129.27'),
(10, 10, '1990-12-08', 'Sri Lanka', 'Nehru Stadium (Margao)', 'c & b S D Anurasiri', 30, 29, '103.45'),
(11, NULL, '1990-12-25', 'Bangladesh', 'Sector 16 Stadium', 'did not bat', NULL, NULL, NULL),
(12, 11, '1990-12-28', 'Sri Lanka', 'Barabati Stadium', 'lbw b A Ranatunga', 4, 11, '36.36'),
(13, 12, '1991-01-04', 'Sri Lanka', 'Eden Gardens', 'lbw b R J Ratnayake', 53, 70, '75.71'),
(14, 13, '1991-10-18', 'Pakistan', 'Sharjah Cricket Stadium', 'not out', 52, 40, '130.0'),
(15, 14, '1991-10-19', 'West Indies', 'Sharjah Cricket Stadium', 'run out', 22, 27, '81.48'),
(16, 15, '1991-10-22', 'West Indies', 'Sharjah Cricket Stadium', 'not out', 11, 27, '40.74'),
(17, 16, '1991-10-23', 'Pakistan', 'Sharjah Cricket Stadium', 'c sub b Saleem Malik', 49, 38, '128.95'),
(18, 17, '1991-10-25', 'Pakistan', 'Sharjah Cricket Stadium', 'lbw b Aaqib Javed', 0, 1, '0.0'),
(19, 18, '1991-11-10', 'South Africa', 'Eden Gardens', 'c R P Snell b A A Donald', 62, 73, '84.93'),
(20, 19, '1991-11-12', 'South Africa', 'Captain Roop Singh Stadium', 'c †D J Richardson b C R Matthews', 4, 8, '50.0'),
(21, 20, '1991-11-14', 'South Africa', 'Jawaharlal Nehru Stadium (Delhi)', 'c S J Cook b A A Donald', 1, 3, '33.33'),
(22, 21, '1991-12-06', 'West Indies', 'WACA Ground', 'c R B Richardson b A C Cummins', 1, 9, '11.11'),
(23, 22, '1991-12-08', 'Australia', 'WACA Ground', 'c P L Taylor b T M Moody', 36, 65, '55.38'),
(24, 23, '1991-12-10', 'Australia', 'Bellerive Oval', 'c S R Waugh b P L Taylor', 57, 107, '53.27'),
(25, 24, '1991-12-14', 'West Indies', 'Adelaide Oval', 'c & b K L T Arthurton', 48, 57, '84.21'),
(26, 25, '1991-12-15', 'Australia', 'Adelaide Oval', 'c D M Jones b S R Waugh', 21, 35, '60.0'),
(27, 26, '1992-01-11', 'West Indies', 'Brisbane Cricket Ground', 'c sub b A C Cummins', 77, 127, '60.63'),
(28, 27, '1992-01-14', 'Australia', 'Sydney Cricket Ground', 'run out', 31, 44, '70.45'),
(29, 28, '1992-01-16', 'West Indies', 'Melbourne Cricket Ground', 'not out', 57, 88, '64.77'),
(30, 29, '1992-01-18', 'Australia', 'Melbourne Cricket Ground', 'c M R Whitney b T M Moody', 4, 10, '40.0'),
(31, 30, '1992-01-20', 'Australia', 'Sydney Cricket Ground', 'c M R Whitney b S R Waugh', 69, 100, '69.0'),
(32, 31, '1992-02-22', 'England', 'WACA Ground', 'c †A J Stewart b I T Botham', 35, 44, '79.55'),
(33, NULL, '1992-02-28', 'Sri Lanka', 'Harrup Park', 'did not bat', NULL, NULL, NULL),
(34, 32, '1992-03-01', 'Australia', 'Brisbane Cricket Ground', 'c S R Waugh b T M Moody', 11, 19, '57.89'),
(35, 33, '1992-03-04', 'Pakistan', 'Sydney Cricket Ground', 'not out', 54, 62, '87.1'),
(36, 34, '1992-03-07', 'Zimbabwe', 'Seddon Park', 'c A D R Campbell b M G Burmester', 81, 77, '105.19'),
(37, 35, '1992-03-10', 'West Indies', 'Basin Reserve', 'c †D Williams b C E L Ambrose', 4, 11, '36.36'),
(38, 36, '1992-03-12', 'New Zealand', 'Carisbrook', 'c †I D S Smith b C Z Harris', 84, 107, '78.5'),
(39, 37, '1992-03-15', 'South Africa', 'Adelaide Oval', 'c K C Wessels b A P Kuiper', 14, 14, '100.0'),
(40, 38, '1992-10-25', 'Zimbabwe', 'Harare Sports Club', 'c D H Brain b G J Crocker', 39, 56, '69.64'),
(41, 39, '1992-12-07', 'South Africa', 'Newlands', 'b B M McMillan', 15, 27, '55.56'),
(42, 40, '1992-12-09', 'South Africa', 'St George''s Park', 'c †D J Richardson b D J Callaghan', 10, 36, '27.78'),
(43, 41, '1992-12-11', 'South Africa', 'SuperSport Park', 'c †D J Richardson b C R Matthews', 22, 24, '91.67'),
(44, 42, '1992-12-13', 'South Africa', 'Wanderers Stadium', 'c B M McMillan b A A Donald', 21, 44, '47.73'),
(45, 43, '1992-12-15', 'South Africa', 'Mangaung Oval', 'b P S De Villiers', 32, 52, '61.54'),
(46, 44, '1992-12-17', 'South Africa', 'Kingsmead', 'c W J Cronje b M W Pringle', 23, 39, '58.97'),
(47, 45, '1992-12-19', 'South Africa', 'Buffalo Park', 'c †D J Richardson b C R Matthews', 21, 38, '55.26'),
(48, 46, '1993-01-18', 'England', 'Sawai Mansingh Stadium', 'not out', 82, 81, '101.23'),
(49, 47, '1993-01-21', 'England', 'Sector 16 Stadium', 'lbw b P A J De Freitas', 1, 5, '20.0'),
(50, 48, '1993-02-26', 'England', 'M Chinnaswamy Stadium', 'c G A Hick b C C Lewis', 3, 6, '50.0'),
(51, 49, '1993-03-01', 'England', 'Keenan Stadium', 'b P W Jarvis', 24, 32, '75.0'),
(52, 50, '1993-03-04', 'England', 'Captain Roop Singh Stadium', 'b P W Jarvis', 5, 6, '83.33'),
(53, 51, '1993-03-05', 'England', 'Captain Roop Singh Stadium', 'c sub b C C Lewis', 34, 30, '113.33'),
(54, 52, '1993-03-19', 'Zimbabwe', 'Nahar Singh Stadium', 'c & b G W Flower', 3, 9, '33.33'),
(55, 53, '1993-03-22', 'Zimbabwe', 'Nehru Stadium (Guwahati)', 'not out', 8, 6, '133.33'),
(56, NULL, '1993-03-25', 'Zimbabwe', 'Nehru Stadium (Pune)', 'did not bat', NULL, NULL, NULL),
(57, 54, '1993-07-25', 'Sri Lanka', 'R Premadasa Stadium', 'c A P Gurusinha b S T Jayasuriya', 21, 39, '53.85'),
(58, 55, '1993-08-12', 'Sri Lanka', 'R Premadasa Stadium', 'run out', 15, 30, '50.0'),
(59, 56, '1993-08-14', 'Sri Lanka', 'Tyronne Fernando Stadium', 'c M Muralitharan b S T Jayasuriya', 25, 39, '64.1'),
(60, 57, '1993-11-07', 'Sri Lanka', 'Green Park', 'not out', 26, 30, '86.67'),
(61, 58, '1993-11-16', 'West Indies', 'Narendra Modi Stadium', 'lbw b C A Walsh', 2, 8, '25.0'),
(62, 59, '1993-11-18', 'Zimbabwe', 'Nehru Stadium (Indore)', 'c & b H H Streak', 24, 16, '150.0'),
(63, 60, '1993-11-22', 'South Africa', 'Punjab Cricket Association IS Bindra Stadium', 'c †D J Richardson b W J Cronje', 3, 25, '12.0'),
(64, 61, '1993-11-24', 'South Africa', 'Eden Gardens', 'c †D J Richardson b R P Snell', 15, 31, '48.39'),
(65, 62, '1993-11-27', 'West Indies', 'Eden Gardens', 'not out', 28, 43, '65.12'),
(66, 63, '1994-02-15', 'Sri Lanka', 'Madhavrao Scindia Cricket Ground', 'c A Ranatunga b R S Kalpage', 1, 5, '20.0'),
(67, 64, '1994-02-18', 'Sri Lanka', 'Lal Bahadur Shastri Stadium', 'not out', 11, 18, '61.11'),
(68, 65, '1994-02-20', 'Sri Lanka', 'Gandhi Stadium', 'run out', 52, 63, '82.54'),
(69, 66, '1994-03-25', 'New Zealand', 'McLean Park', 'c K R Rutherford b D J Nash', 15, 19, '78.95'),
(70, 67, '1994-03-27', 'New Zealand', 'Eden Park', 'c & b M N Hart', 82, 49, '167.35'),
(71, 68, '1994-03-30', 'New Zealand', 'Basin Reserve', 'lbw b G R Larsen', 63, 75, '84.0'),
(72, 69, '1994-04-02', 'New Zealand', 'AMI Stadium', 'b G R Larsen', 40, 26, '153.85'),
(73, 70, '1994-04-13', 'United Arab Emirates', 'Sharjah Cricket Stadium', 'c †Imtiaz Abbasi b Sultan Zarawani', 63, 77, '81.82'),
(74, 71, '1994-04-15', 'Pakistan', 'Sharjah Cricket Stadium', 'c Basit Ali b Akram Raza', 73, 64, '114.06'),
(75, 72, '1994-04-19', 'Australia', 'Sharjah Cricket Stadium', 'c M A Taylor b G D McGrath', 6, 7, '85.71'),
(76, 73, '1994-04-22', 'Pakistan', 'Sharjah Cricket Stadium', 'c Aamir Sohail b AtaUrRehman', 24, 26, '92.31'),
(77, 74, '1994-09-04', 'Sri Lanka', 'R Premadasa Stadium', 'not out', 11, 16, '68.75'),
(78, 75, '1994-09-05', 'Sri Lanka', 'R Premadasa Stadium', 'c H D P K Dharmasena b G P Wickramasinghe', 6, 5, '120.0'),
(79, 76, '1994-09-09', 'Australia', 'R Premadasa Stadium', 'b C J McDermott', 110, 130, '84.62'),
(80, 77, '1994-09-17', 'Sri Lanka', 'Sinhalese Sports Club Ground', 'c P A de Silva b W P U J C Vaas', 0, 2, '0.0'),
(81, 78, '1994-10-17', 'West Indies', 'Nahar Singh Stadium', 'c B C Lara b C A Walsh', 0, 4, '0.0'),
(82, 79, '1994-10-20', 'West Indies', 'Wankhede Stadium', 'c C L Hooper b C E Cuffy', 0, 4, '0.0'),
(83, 80, '1994-10-23', 'West Indies', 'MA Chidambaram Stadium', 'c C L Hooper b A C Cummins', 8, 24, '33.33'),
(84, 81, '1994-10-28', 'New Zealand', 'Reliance Stadium', 'run out', 115, 136, '84.56'),
(85, 82, '1994-10-30', 'West Indies', 'Green Park', 'b A C Cummins', 34, 47, '72.34'),
(86, 83, '1994-11-03', 'New Zealand', 'Arun Jaitley Stadium', 'b M N Hart', 62, 54, '114.81'),
(87, 84, '1994-11-05', 'West Indies', 'Eden Gardens', 'c S C Williams b C E Cuffy', 66, 68, '97.06'),
(88, 85, '1994-11-07', 'West Indies', 'Indira Priyadarshini Stadium', 'c A C Cummins b C L Hooper', 54, 64, '84.38'),
(89, 86, '1994-11-09', 'West Indies', 'Barabati Stadium', 'b P V Simmons', 88, 112, '78.57'),
(90, 87, '1994-11-11', 'West Indies', 'Sawai Mansingh Stadium', 'c †J C Adams b B S Browne', 105, 134, '78.36'),
(91, 88, '1995-02-16', 'New Zealand', 'McLean Park', 'c S A Thomson b D K Morrison', 13, 15, '86.67'),
(92, 89, '1995-02-18', 'South Africa', 'Seddon Park', 'c P L Symcox b W J Cronje', 37, 51, '72.55'),
(93, 90, '1995-02-22', 'Australia', 'Carisbrook', 'c M A Taylor b J Angel', 47, 40, '117.5'),
(94, 91, '1995-04-05', 'Bangladesh', 'Sharjah Cricket Stadium', 'b Mohammad Rafique', 48, 30, '160.0'),
(95, 92, '1995-04-07', 'Pakistan', 'Sharjah Cricket Stadium', 'c †Moin Khan b Aaqib Javed', 4, 9, '44.44'),
(96, 93, '1995-04-09', 'Sri Lanka', 'Sharjah Cricket Stadium', 'not out', 112, 107, '104.67'),
(97, 94, '1995-04-14', 'Sri Lanka', 'Sharjah Cricket Stadium', 'c S T Jayasuriya b C P H Ramanayake', 41, 41, '100.0'),
(98, 95, '1995-11-15', 'New Zealand', 'Keenan Stadium', 'c M J Greatbatch b D K Morrison', 30, 20, '150.0'),
(99, 96, '1995-11-18', 'New Zealand', 'Gandhi Sports Complex Ground', 'c †L K Germon b S A Thomson', 39, 51, '76.47'),
(100, 97, '1995-11-24', 'New Zealand', 'Nehru Stadium (Pune)', 'c G R Larsen b D K Morrison', 7, 11, '63.64'),
(101, 98, '1995-11-26', 'New Zealand', 'Vidarbha Cricket Association Ground', 'run out', 65, 59, '110.17'),
(102, 99, '1995-11-29', 'New Zealand', 'Brabourne Stadium', 'b D K Morrison', 1, 4, '25.0'),
(103, 100, '1996-02-18', 'Kenya', 'Barabati Stadium', 'not out', 127, 138, '92.03'),
(104, 101, '1996-02-21', 'West Indies', 'Captain Roop Singh Stadium', 'run out', 70, 91, '76.92'),
(105, 102, '1996-02-27', 'Australia', 'Wankhede Stadium', 'st I A Healy b M E Waugh', 90, 84, '107.14'),
(106, 103, '1996-03-02', 'Sri Lanka', 'Arun Jaitley Stadium', 'run out', 137, 137, '100.0'),
(107, 104, '1996-03-06', 'Zimbabwe', 'Green Park', 'b H H Streak', 3, 12, '25.0'),
(108, 105, '1996-03-09', 'Pakistan', 'M Chinnaswamy Stadium', 'b AtaUrRehman', 31, 59, '52.54'),
(109, 106, '1996-03-13', 'Sri Lanka', 'Eden Gardens', 'st R S Kaluwitharana b S T Jayasuriya', 65, 88, '73.86'),
(110, 107, '1996-04-03', 'Sri Lanka', 'Padang Cricket Ground', 'c S T Jayasuriya b G P Wickramasinghe', 28, 31, '90.32'),
(111, 108, '1996-04-05', 'Pakistan', 'Padang Cricket Ground', 'st Rashid Latif b Saqlain Mushtaq', 100, 111, '90.09'),
(112, 109, '1996-04-12', 'Pakistan', 'Sharjah Cricket Stadium', 'c Saeed Anwar b Aaqib Javed', 1, 5, '20.0'),
(113, 110, '1996-04-14', 'South Africa', 'Sharjah Cricket Stadium', 'c G Kirsten b P S De Villiers', 2, 15, '13.33'),
(114, 111, '1996-04-15', 'Pakistan', 'Sharjah Cricket Stadium', 'c Aamir Sohail b Waqar Younis', 118, 140, '84.29'),
(115, 112, '1996-04-17', 'South Africa', 'Sharjah Cricket Stadium', 'c G Kirsten b P S De Villiers', 17, 26, '65.38'),
(116, 113, '1996-04-19', 'South Africa', 'Sharjah Cricket Stadium', 'run out', 57, 71, '80.28'),
(117, 114, '1996-05-23', 'England', 'Kennington Oval', 'lbw b P J Martin', 30, 19, '157.89'),
(118, 115, '1996-05-25', 'England', 'Headingley', 'run out', 6, 19, '31.58'),
(119, 116, '1996-05-26', 'England', 'Old Trafford', 'c G A Hick b D G Cork', 1, 11, '9.09'),
(120, 117, '1996-08-28', 'Sri Lanka', 'R Premadasa Stadium', 'run out', 110, 138, '79.71'),
(121, 118, '1996-09-01', 'Zimbabwe', 'Sinhalese Sports Club Ground', 'c B C Strang b H H Streak', 40, 46, '86.96'),
(122, 119, '1996-09-06', 'Australia', 'Sinhalese Sports Club Ground', 'c S R Waugh b G D McGrath', 7, 11, '63.64'),
(123, 120, '1996-09-16', 'Pakistan', 'Cricket, Skating & Curling Club', 'not out', 89, 89, '100.0'),
(124, 121, '1996-09-17', 'Pakistan', 'Cricket, Skating & Curling Club', 'c Wasim Akram b Azhar Mahmood', 20, 23, '86.96'),
(125, 122, '1996-09-18', 'Pakistan', 'Cricket, Skating & Curling Club', 'c Aamir Sohail b Wasim Akram', 2, 13, '15.38'),
(126, 123, '1996-09-21', 'Pakistan', 'Cricket, Skating & Curling Club', 'c Saleem Malik b Wasim Akram', 3, 9, '33.33'),
(127, 124, '1996-09-23', 'Pakistan', 'Cricket, Skating & Curling Club', 'run out', 23, 44, '52.27'),
(128, 125, '1996-10-17', 'South Africa', 'Lal Bahadur Shastri Stadium', 'c D J Cullinan b P S De Villiers', 11, 8, '137.5'),
(129, 126, '1996-10-21', 'Australia', 'M Chinnaswamy Stadium', 'lbw b S R Waugh', 88, 111, '79.28'),
(130, 127, '1996-10-23', 'South Africa', 'Sawai Mansingh Stadium', 'c G Kirsten b B M McMillan', 64, 93, '68.82'),
(131, 128, '1996-10-29', 'South Africa', 'Madhavrao Scindia Cricket Ground', 'lbw b A A Donald', 28, 38, '73.68'),
(132, 129, '1996-11-03', 'Australia', 'Punjab Cricket Association IS Bindra Stadium', 'c S G Law b M E Waugh', 62, 60, '103.33'),
(133, 130, '1996-11-06', 'South Africa', 'Wankhede Stadium', 'c W J Cronje b N Boje', 67, 88, '76.14'),
(134, 131, '1996-12-14', 'South Africa', 'Wankhede Stadium', 'st G Kirsten b N Boje', 114, 126, '90.48'),
(135, 132, '1997-01-23', 'South Africa', 'Mangaung Oval', 'b S M Pollock', 0, 4, '0.0'),
(136, 133, '1997-01-27', 'Zimbabwe', 'Boland Park', 'c A D R Campbell b E A Brandes', 6, 8, '75.0'),
(137, 134, '1997-02-02', 'South Africa', 'St George''s Park', 'b A A Donald', 1, 14, '7.14'),
(138, 135, '1997-02-04', 'South Africa', 'Buffalo Park', 'c D J Cullinan b L Klusener', 14, 24, '58.33'),
(139, 136, '1997-02-07', 'Zimbabwe', 'SuperSport Park', 'c A C Waller b A D R Campbell', 41, 56, '73.21'),
(140, 137, '1997-02-09', 'Zimbabwe', 'Willowmoore Park', 'c A D R Campbell b C N Evans', 104, 97, '107.22'),
(141, 138, '1997-02-12', 'South Africa', 'Kingsmead', 'c J N Rhodes b L Klusener', 32, 27, '118.52'),
(142, 139, '1997-02-13', 'South Africa', 'Kingsmead', 'c R E Bryson b W J Cronje', 45, 33, '136.36'),
(143, 140, '1997-02-15', 'Zimbabwe', 'Queens Sports Club', 'c G W Flower b E A Brandes', 13, 15, '86.67'),
(144, 141, '1997-04-26', 'West Indies', 'Queen''s Park Oval', 'c †C O Browne b C E L Ambrose', 44, 43, '102.33'),
(145, 142, '1997-04-27', 'West Indies', 'Queen''s Park Oval', 'not out', 65, 70, '92.86'),
(146, 143, '1997-04-30', 'West Indies', 'Arnos Vale Ground', 'b C A Walsh', 9, 15, '60.0'),
(147, 144, '1997-05-03', 'West Indies', 'Kensington Oval', 'c B C Lara b C A Walsh', 1, 11, '9.09'),
(148, 145, '1997-05-14', 'New Zealand', 'M Chinnaswamy Stadium', 'b N J Astle', 117, 137, '85.4'),
(149, 146, '1997-05-17', 'Sri Lanka', 'Wankhede Stadium', 'c H D P K Dharmasena b K S C de Silva', 2, 4, '50.0'),
(150, 147, '1997-05-21', 'Pakistan', 'MA Chidambaram Stadium', 'c InzamamulHaq b Aaqib Javed', 4, 7, '57.14'),
(151, 148, '1997-07-18', 'Sri Lanka', 'R Premadasa Stadium', 'b W P U J C Vaas', 21, 28, '75.0'),
(152, NULL, '1997-07-20', 'Pakistan', 'Sinhalese Sports Club Ground', 'did not bat', NULL, NULL, NULL),
(153, 149, '1997-07-24', 'Bangladesh', 'Sinhalese Sports Club Ground', 'b Enamul Haque', 28, 21, '133.33'),
(154, 150, '1997-07-26', 'Sri Lanka', 'R Premadasa Stadium', 'c R S Kalpage b M Muralitharan', 53, 67, '79.1'),
(155, 151, '1997-08-17', 'Sri Lanka', 'R Premadasa Stadium', 'c M Muralitharan b W P U J C Vaas', 27, 28, '96.43'),
(156, 152, '1997-08-20', 'Sri Lanka', 'R Premadasa Stadium', 'lbw b W P U J C Vaas', 6, 6, '100.0'),
(157, 153, '1997-08-23', 'Sri Lanka', 'Sinhalese Sports Club Ground', 'c A Ranatunga b D K Liyanage', 27, 31, '87.1'),
(158, 154, '1997-08-24', 'Sri Lanka', 'Sinhalese Sports Club Ground', 'c †S K L de Silva b K S C de Silva', 39, 32, '121.88'),
(159, 155, '1997-09-13', 'Pakistan', 'Cricket, Skating & Curling Club', 'c Mohammad Akram b Azhar Mahmood', 17, 54, '31.48'),
(160, 156, '1997-09-14', 'Pakistan', 'Cricket, Skating & Curling Club', 'not out', 25, 45, '55.56'),
(161, NULL, '1997-09-17', 'Pakistan', 'Cricket, Skating & Curling Club', 'did not bat', NULL, NULL, NULL),
(162, 157, '1997-09-18', 'Pakistan', 'Cricket, Skating & Curling Club', 'c †Moin Khan b Mohammad Akram', 0, 10, '0.0'),
(163, 158, '1997-09-20', 'Pakistan', 'Cricket, Skating & Curling Club', 'c †Moin Khan b Shahid Nazir', 6, 7, '85.71'),
(164, 159, '1997-09-21', 'Pakistan', 'Cricket, Skating & Curling Club', 'lbw b Azhar Mahmood', 51, 64, '79.69'),
(165, 160, '1997-09-28', 'Pakistan', 'Niaz Stadium', 'b Waqar Younis', 2, 11, '18.18'),
(166, 161, '1997-09-30', 'Pakistan', 'National Stadium (Karachi)', 'c †Moin Khan b Azhar Mahmood', 21, 18, '116.67'),
(167, 162, '1997-10-02', 'Pakistan', 'Gaddafi Stadium', 'c InzamamulHaq b Aaqib Javed', 7, 11, '63.64'),
(168, 163, '1997-12-11', 'England', 'Sharjah Cricket Stadium', 'st A J Stewart b M V Fleming', 91, 87, '104.6'),
(169, 164, '1997-12-14', 'Pakistan', 'Sharjah Cricket Stadium', 'c InzamamulHaq b Manzoor Akhtar', 3, 4, '75.0'),
(170, 165, '1997-12-16', 'West Indies', 'Sharjah Cricket Stadium', 'run out', 1, 2, '50.0'),
(171, 166, '1997-12-22', 'Sri Lanka', 'Nehru Stadium (Guwahati)', 'not out', 82, 86, '95.35'),
(172, NULL, '1997-12-25', 'Sri Lanka', 'Nehru Stadium (Indore)', 'did not bat', NULL, NULL, NULL),
(173, 167, '1997-12-28', 'Sri Lanka', 'Nehru Stadium (Margao)', 'c K S C de Silva b M Muralitharan', 6, 13, '46.15'),
(174, 168, '1998-01-10', 'Bangladesh', 'Bangabandhu National Stadium', 'c & b Mohammad Rafique', 54, 76, '71.05'),
(175, 169, '1998-01-11', 'Pakistan', 'Bangabandhu National Stadium', 'st Rashid Latif b Saqlain Mushtaq', 67, 44, '152.27'),
(176, 170, '1998-01-14', 'Pakistan', 'Bangabandhu National Stadium', 'b Shahid Afridi', 95, 78, '121.79'),
(177, 171, '1998-01-16', 'Pakistan', 'Bangabandhu National Stadium', 'b Azhar Mahmood', 1, 6, '16.67'),
(178, 172, '1998-01-18', 'Pakistan', 'Bangabandhu National Stadium', 'c Azhar Mahmood b Shahid Afridi', 41, 26, '157.69'),
(179, 173, '1998-04-01', 'Australia', 'Nehru Stadium (Kochi)', 'c R T Ponting b M S Kasprowicz', 8, 11, '72.73'),
(180, 174, '1998-04-05', 'Zimbabwe', 'Reliance Stadium', 'run out', 5, 17, '29.41'),
(181, 175, '1998-04-07', 'Australia', 'Green Park', 'c sub b S K Warne', 100, 89, '112.36'),
(182, 176, '1998-04-09', 'Zimbabwe', 'Barabati Stadium', 'c †A Flower b M Mbangwa', 1, 2, '50.0'),
(183, 177, '1998-04-14', 'Australia', 'Arun Jaitley Stadium', 'c †A C Gilchrist b D W Fleming', 15, 24, '62.5'),
(184, 178, '1998-04-17', 'New Zealand', 'Sharjah Cricket Stadium', 'c S B Doull b C Z Harris', 40, 41, '97.56'),
(185, 179, '1998-04-19', 'Australia', 'Sharjah Cricket Stadium', 'c †A C Gilchrist b D W Fleming', 80, 72, '111.11'),
(186, 180, '1998-04-20', 'New Zealand', 'Sharjah Cricket Stadium', 'run out', 38, 58, '65.52'),
(187, 181, '1998-04-22', 'Australia', 'Sharjah Cricket Stadium', 'c †A C Gilchrist b D W Fleming', 143, 131, '109.16'),
(188, 182, '1998-04-24', 'Australia', 'Sharjah Cricket Stadium', 'lbw b M S Kasprowicz', 134, 131, '102.29'),
(189, 183, '1998-05-25', 'Bangladesh', 'Wankhede Stadium', 'c Aminul Islam b Athar Ali Khan', 33, 29, '113.79'),
(190, 184, '1998-05-28', 'Kenya', 'Captain Roop Singh Stadium', 'c A Y A Karim b M A Suji', 18, 25, '72.0'),
(191, 185, '1998-05-31', 'Kenya', 'Eden Gardens', 'not out', 100, 103, '97.09'),
(192, 186, '1998-06-19', 'Sri Lanka', 'R Premadasa Stadium', 'c M S Atapattu b M Muralitharan', 65, 50, '130.0'),
(193, 187, '1998-06-23', 'New Zealand', 'R Premadasa Stadium', 'c & b C Z Harris', 53, 36, '147.22'),
(194, 188, '1998-07-01', 'Sri Lanka', 'Sinhalese Sports Club Ground', 'c & b H D P K Dharmasena', 17, 16, '106.25'),
(195, NULL, '1998-07-03', 'New Zealand', 'Sinhalese Sports Club Ground', 'did not bat', NULL, NULL, NULL),
(196, 189, '1998-07-07', 'Sri Lanka', 'R Premadasa Stadium', 'st R S Kaluwitharana b S T Jayasuriya', 128, 131, '97.71'),
(197, 190, '1998-09-20', 'Pakistan', 'Cricket, Skating & Curling Club', 'c InzamamulHaq b Aamir Sohail', 77, 109, '70.64'),
(198, 191, '1998-09-26', 'Zimbabwe', 'Queens Sports Club', 'not out', 127, 130, '97.69'),
(199, 192, '1998-09-27', 'Zimbabwe', 'Queens Sports Club', 'c C B Wishart b M L Nkala', 29, 21, '138.1'),
(200, 193, '1998-09-30', 'Zimbabwe', 'Harare Sports Club', 'c C N Evans b H H Streak', 2, 6, '33.33'),
(201, 194, '1998-10-28', 'Australia', 'Bangabandhu National Stadium', 'run out', 141, 128, '110.16'),
(202, 195, '1998-10-31', 'West Indies', 'Bangabandhu National Stadium', 'c C L Hooper b M V Dillon', 8, 14, '57.14'),
(203, 196, '1998-11-06', 'Sri Lanka', 'Sharjah Cricket Stadium', 'c †R S Kaluwitharana b W P U J C Vaas', 3, 6, '50.0'),
(204, 197, '1998-11-08', 'Zimbabwe', 'Sharjah Cricket Stadium', 'not out', 118, 112, '105.36'),
(205, 198, '1998-11-09', 'Sri Lanka', 'Sharjah Cricket Stadium', 'c M S Atapattu b G P Wickramasinghe', 18, 28, '64.29'),
(206, 199, '1998-11-11', 'Zimbabwe', 'Sharjah Cricket Stadium', 'c G W Flower b H K Olonga', 11, 12, '91.67'),
(207, 200, '1998-11-13', 'Zimbabwe', 'Sharjah Cricket Stadium', 'not out', 124, 92, '134.78'),
(208, 201, '1999-01-09', 'New Zealand', 'Owen Delany Park', 'c C Z Harris b C L Cairns', 0, 5, '0.0'),
(209, 202, '1999-01-12', 'New Zealand', 'McLean Park', 'c B A Young b D J Nash', 23, 19, '121.05'),
(210, 203, '1999-01-14', 'New Zealand', 'Basin Reserve', 'st A C Parore b G R Larsen', 45, 42, '107.14'),
(211, 204, '1999-01-16', 'New Zealand', 'Eden Park', 'lbw b C L Cairns', 5, 12, '41.67'),
(212, 205, '1999-05-15', 'South Africa', 'County Ground (Hove)', 'c †M V Boucher b L Klusener', 28, 46, '60.87'),
(213, 206, '1999-05-23', 'Kenya', 'County Ground (Bristol)', 'not out', 140, 101, '138.61'),
(214, 207, '1999-05-26', 'Sri Lanka', 'The Cooper Associates County Ground', 'b S T Jayasuriya', 2, 3, '66.67'),
(215, 208, '1999-05-29', 'England', 'Edgbaston', 'c G A Hick b M A Ealham', 22, 40, '55.0'),
(216, 209, '1999-06-04', 'Australia', 'Kennington Oval', 'c †A C Gilchrist b G D McGrath', 0, 4, '0.0'),
(217, 210, '1999-06-08', 'Pakistan', 'Old Trafford', 'c Saqlain Mushtaq b Azhar Mahmood', 45, 65, '69.23'),
(218, 211, '1999-06-12', 'New Zealand', 'Trent Bridge', 'b D J Nash', 16, 22, '72.73'),
(219, 212, '1999-08-23', 'Australia', 'Galle International Stadium', 'c D S Lehmann b T M Moody', 14, 33, '42.42'),
(220, 213, '1999-08-25', 'Sri Lanka', 'R Premadasa Stadium', 'run out', 37, 58, '63.79'),
(221, 214, '1999-08-29', 'Sri Lanka', 'Sinhalese Sports Club Ground', 'c M Muralitharan b D N T Zoysa', 120, 141, '85.11'),
(222, 215, '1999-09-04', 'Zimbabwe', 'Kallang Ground', 'c S V Carlisle b A R Whittall', 85, 72, '118.06'),
(223, 216, '1999-09-07', 'West Indies', 'Kallang Ground', 'c †R D Jacobs b C A Walsh', 40, 65, '61.54'),
(224, 217, '1999-09-08', 'West Indies', 'Kallang Ground', 'c H R Bryan b C A Walsh', 0, 6, '0.0'),
(225, 218, '1999-11-05', 'New Zealand', 'Madhavrao Scindia Cricket Ground', 'c C L Cairns b S B Styris', 32, 31, '103.23'),
(226, 219, '1999-11-08', 'New Zealand', 'Lal Bahadur Shastri Stadium', 'not out', 186, 150, '124.0'),
(227, 220, '1999-11-11', 'New Zealand', 'Captain Roop Singh Stadium', 'c S P Fleming b C J Drum', 1, 23, '4.35'),
(228, 221, '1999-11-14', 'New Zealand', 'Nehru Stadium (Guwahati)', 'c C M Spearman b C J Drum', 2, 10, '20.0'),
(229, 222, '1999-11-17', 'New Zealand', 'Arun Jaitley Stadium', 'c & b D L Vettori', 0, 3, '0.0'),
(230, 223, '2000-01-10', 'Pakistan', 'Brisbane Cricket Ground', 'b Abdul Razzaq', 13, 26, '50.0'),
(231, 224, '2000-01-12', 'Australia', 'Melbourne Cricket Ground', 'run out', 12, 11, '109.09'),
(232, 225, '2000-01-14', 'Australia', 'Sydney Cricket Ground', 'c †A C Gilchrist b G D McGrath', 1, 11, '9.09'),
(233, 226, '2000-01-21', 'Pakistan', 'Bellerive Oval', 'b Abdul Razzaq', 93, 103, '90.29'),
(234, 227, '2000-01-25', 'Pakistan', 'Adelaide Oval', 'c †Moin Khan b Abdul Razzaq', 41, 46, '89.13'),
(235, 228, '2000-01-26', 'Australia', 'Adelaide Oval', 'c S C G MacGill b B Lee', 18, 28, '64.29'),
(236, 229, '2000-01-28', 'Pakistan', 'WACA Ground', 'c †Moin Khan b Waqar Younis', 17, 14, '121.43'),
(237, 230, '2000-01-30', 'Australia', 'WACA Ground', 'b D W Fleming', 3, 21, '14.29'),
(238, 231, '2000-03-09', 'South Africa', 'Nehru Stadium (Kochi)', 'c H S Williams b M Hayward', 26, 25, '104.0'),
(239, 232, '2000-03-12', 'South Africa', 'Keenan Stadium', 'c W J Cronje b S M Pollock', 21, 31, '67.74'),
(240, 233, '2000-03-15', 'South Africa', 'Nahar Singh Stadium', 'lbw b S M Pollock', 12, 28, '42.86'),
(241, 234, '2000-03-17', 'South Africa', 'Reliance Stadium', 'c S Elworthy b J H Kallis', 122, 138, '88.41'),
(242, 235, '2000-03-19', 'South Africa', 'Vidarbha Cricket Association Ground', 'c S Elworthy b D N Crookes', 93, 89, '104.49'),
(243, 236, '2000-03-22', 'South Africa', 'Sharjah Cricket Stadium', 'b S M Pollock', 5, 8, '62.5'),
(244, 237, '2000-03-23', 'Pakistan', 'Sharjah Cricket Stadium', 'lbw b Shoaib Akhtar', 11, 28, '39.29'),
(245, 238, '2000-03-26', 'Pakistan', 'Sharjah Cricket Stadium', 'b Wasim Akram', 10, 18, '55.56'),
(246, 239, '2000-03-27', 'South Africa', 'Sharjah Cricket Stadium', 'run out', 39, 68, '57.35'),
(247, 240, '2000-05-30', 'Bangladesh', 'Bangabandhu National Stadium', 'c Habibul Bashar b Mushfiqur Rahman', 36, 25, '144.0'),
(248, 241, '2000-06-01', 'Sri Lanka', 'Bangabandhu National Stadium', 'c D P M D Jayawardene b K Weeraratne', 93, 95, '97.89'),
(249, 242, '2000-06-03', 'Pakistan', 'Bangabandhu National Stadium', 'lbw b Abdul Razzaq', 25, 30, '83.33'),
(250, 243, '2000-10-03', 'Kenya', 'Gymkhana Club Ground', 'lbw b A O Suji', 25, 35, '71.43'),
(251, 244, '2000-10-07', 'Australia', 'Gymkhana Club Ground', 'c D R Martyn b B Lee', 38, 37, '102.7'),
(252, 245, '2000-10-13', 'South Africa', 'Gymkhana Club Ground', 'c L Klusener b J H Kallis', 39, 50, '78.0'),
(253, 246, '2000-10-15', 'New Zealand', 'Gymkhana Club Ground', 'run out', 69, 83, '83.13'),
(254, 247, '2000-10-20', 'Sri Lanka', 'Sharjah Cricket Stadium', 'run out', 101, 140, '72.14'),
(255, 248, '2000-10-22', 'Zimbabwe', 'Sharjah Cricket Stadium', 'c †A Flower b H H Streak', 8, 15, '53.33'),
(256, 249, '2000-10-26', 'Zimbabwe', 'Sharjah Cricket Stadium', 'c P A Strang b T J Friend', 4, 10, '40.0'),
(257, 250, '2000-10-27', 'Sri Lanka', 'Sharjah Cricket Stadium', 'c W P U J C Vaas b M Muralitharan', 61, 54, '112.96'),
(258, 251, '2000-10-29', 'Sri Lanka', 'Sharjah Cricket Stadium', 'c & b W P U J C Vaas', 5, 11, '45.45'),
(259, 252, '2000-12-02', 'Zimbabwe', 'Barabati Stadium', 'c H H Streak b D P Viljoen', 44, 49, '89.8'),
(260, 253, '2000-12-05', 'Zimbabwe', 'Narendra Modi Stadium', 'c †A Flower b T J Friend', 8, 20, '40.0'),
(261, 254, '2000-12-08', 'Zimbabwe', 'Barkatullah Khan Stadium', 'c M L Nkala b H H Streak', 146, 153, '95.42'),
(262, 255, '2000-12-11', 'Zimbabwe', 'Green Park', 'lbw b T J Friend', 62, 86, '72.09'),
(263, 256, '2000-12-14', 'Zimbabwe', 'Madhavrao Scindia Cricket Ground', 'b M L Nkala', 27, 38, '71.05'),
(264, 257, '2001-03-25', 'Australia', 'M Chinnaswamy Stadium', 'run out', 35, 26, '134.62'),
(265, 258, '2001-03-28', 'Australia', 'Nehru Stadium (Pune)', 'c D S Lehmann b D W Fleming', 32, 29, '110.34'),
(266, 259, '2001-03-31', 'Australia', 'Nehru Stadium (Indore)', 'c D W Fleming b G D McGrath', 139, 125, '111.2'),
(267, 260, '2001-04-03', 'Australia', 'Indira Priyadarshini Stadium', 'c S R Waugh b N W Bracken', 62, 38, '163.16'),
(268, 261, '2001-04-06', 'Australia', 'Nehru Stadium (Margao)', 'c †A C Gilchrist b N W Bracken', 12, 15, '80.0'),
(269, 262, '2001-06-24', 'Zimbabwe', 'Harare Sports Club', 'not out', 70, 70, '100.0'),
(270, 263, '2001-06-27', 'Zimbabwe', 'Queens Sports Club', 'c G W Flower b B C Strang', 9, 27, '33.33'),
(271, 264, '2001-06-30', 'West Indies', 'Queens Sports Club', 'not out', 81, 110, '73.64'),
(272, 265, '2001-07-04', 'West Indies', 'Harare Sports Club', 'not out', 122, 131, '93.13'),
(273, 266, '2001-07-07', 'West Indies', 'Harare Sports Club', 'c D Ganga b C D Collymore', 0, 4, '0.0'),
(274, 267, '2001-10-05', 'South Africa', 'Wanderers Stadium', 'c H H Gibbs b J H Kallis', 101, 129, '78.29'),
(275, 268, '2001-10-10', 'South Africa', 'SuperSport Park', 'c A Nel b M Ntini', 38, 57, '66.67'),
(276, NULL, '2001-10-12', 'Kenya', 'Mangaung Oval', 'did not bat', NULL, NULL, NULL),
(277, 269, '2001-10-17', 'Kenya', 'St George''s Park', 'b J O Angara', 3, 20, '15.0'),
(278, 270, '2001-10-19', 'South Africa', 'Buffalo Park', 'b J H Kallis', 37, 35, '105.71'),
(279, 271, '2001-10-24', 'Kenya', 'Boland Park', 'c M O Odumbe b T Odoyo', 146, 132, '110.61'),
(280, 272, '2001-10-26', 'South Africa', 'Kingsmead', 'b M Hayward', 17, 42, '40.48'),
(281, 273, '2002-01-19', 'England', 'Eden Gardens', 'b A Flintoff', 36, 43, '83.72'),
(282, 274, '2002-01-22', 'England', 'Barabati Stadium', 'run out', 45, 60, '75.0'),
(283, 275, '2002-01-25', 'England', 'MA Chidambaram Stadium', 'lbw b J N Snape', 68, 79, '86.08'),
(284, 276, '2002-01-28', 'England', 'Green Park', 'not out', 87, 67, '129.85'),
(285, 277, '2002-01-31', 'England', 'Arun Jaitley Stadium', 'c †J S Foster b A R Caddick', 18, 16, '112.5'),
(286, 278, '2002-02-03', 'England', 'Wankhede Stadium', 'c †J S Foster b D Gough', 12, 18, '66.67'),
(287, 279, '2002-05-29', 'West Indies', 'Kensington Oval', 'not out', 34, 45, '75.56'),
(288, 280, '2002-06-02', 'West Indies', 'Queen''s Park Oval', 'b M V Dillon', 65, 70, '92.86'),
(289, 281, '2002-06-29', 'England', 'Lord''s', 'lbw b R C Irani', 1, 9, '11.11'),
(290, 282, '2002-06-30', 'Sri Lanka', 'Kennington Oval', 'c †R S Kaluwitharana b D N T Zoysa', 49, 70, '70.0'),
(291, 283, '2002-07-04', 'England', 'Riverside Ground', 'not out', 105, 108, '97.22'),
(292, 284, '2002-07-06', 'Sri Lanka', 'Edgbaston', 'c M S Atapattu b C R D Fernando', 19, 25, '76.0'),
(293, 285, '2002-07-09', 'England', 'Kennington Oval', 'c †A J Stewart b M J Hoggard', 36, 29, '124.14'),
(294, 286, '2002-07-11', 'Sri Lanka', 'County Ground (Bristol)', 'c U D U Chandana b W P U J C Vaas', 113, 102, '110.78'),
(295, 287, '2002-07-13', 'England', 'Lord''s', 'b A F Giles', 14, 19, '73.68'),
(296, 288, '2002-09-14', 'Zimbabwe', 'R Premadasa Stadium', 'c A D R Campbell b D T Hondo', 7, 16, '43.75'),
(297, 289, '2002-09-22', 'England', 'R Premadasa Stadium', 'not out', 9, 20, '45.0'),
(298, 290, '2002-09-25', 'South Africa', 'R Premadasa Stadium', 'run out', 16, 29, '55.17'),
(299, NULL, '2002-09-29', 'Sri Lanka', 'R Premadasa Stadium', 'did not bat', NULL, NULL, NULL),
(300, 291, '2002-09-30', 'Sri Lanka', 'R Premadasa Stadium', 'not out', 7, 22, '31.82'),
(301, 292, '2003-01-08', 'New Zealand', 'Westpac Stadium', 'lbw b S E Bond', 0, 10, '0.0'),
(302, 293, '2003-01-11', 'New Zealand', 'Eden Park', 'c †B B McCullum b D R Tuffey', 1, 13, '7.69'),
(303, 294, '2003-01-14', 'New Zealand', 'Seddon Park', 'c S P Fleming b D R Tuffey', 1, 6, '16.67'),
(304, 295, '2003-02-12', 'Netherlands', 'Boland Park', 'c †J Smits b T B M de Leede', 52, 72, '72.22'),
(305, 296, '2003-02-15', 'Australia', 'SuperSport Park', 'lbw b J N Gillespie', 36, 59, '61.02'),
(306, 297, '2003-02-19', 'Zimbabwe', 'Harare Sports Club', 'b G W Flower', 81, 91, '89.01'),
(307, 298, '2003-02-23', 'Namibia', 'City Oval', 'b R J van Vuuren', 152, 151, '100.66'),
(308, 299, '2003-02-26', 'England', 'Kingsmead', 'c P D Collingwood b A Flintoff', 50, 52, '96.15'),
(309, 300, '2003-03-01', 'Pakistan', 'SuperSport Park', 'c Younis Khan b Shoaib Akhtar', 98, 75, '130.67'),
(310, 301, '2003-03-07', 'Kenya', 'Newlands', 'c A O Suji b M A Suji', 5, 12, '41.67'),
(311, 302, '2003-03-10', 'Sri Lanka', 'Wanderers Stadium', 'c †K C Sangakkara b P A de Silva', 97, 120, '80.83'),
(312, 303, '2003-03-14', 'New Zealand', 'SuperSport Park', 'c J D P Oram b D R Tuffey', 15, 16, '93.75'),
(313, 304, '2003-03-20', 'Kenya', 'Kingsmead', 'c D O Obuya b S O Tikolo', 83, 101, '82.18'),
(314, 305, '2003-03-23', 'Australia', 'Wanderers Stadium', 'c & b G D McGrath', 4, 5, '80.0'),
(315, 306, '2003-10-23', 'New Zealand', 'MA Chidambaram Stadium', 'not out', 48, 66, '72.73'),
(316, 307, '2003-10-26', 'Australia', 'Captain Roop Singh Stadium', 'c †A C Gilchrist b N W Bracken', 100, 119, '84.03'),
(317, 308, '2003-11-01', 'Australia', 'Wankhede Stadium', 'b M J Clarke', 68, 76, '89.47'),
(318, 309, '2003-11-06', 'New Zealand', 'Barabati Stadium', 'lbw b K D Mills', 14, 14, '100.0'),
(319, 310, '2003-11-12', 'Australia', 'M Chinnaswamy Stadium', 'b I J Harvey', 89, 91, '97.8'),
(320, 311, '2003-11-15', 'New Zealand', 'Lal Bahadur Shastri Stadium', 'c J D P Oram b C Z Harris', 102, 91, '112.09'),
(321, 312, '2003-11-18', 'Australia', 'Eden Gardens', 'b A J Bichel', 45, 66, '68.18'),
(322, 313, '2004-01-09', 'Australia', 'Melbourne Cricket Ground', 'c R T Ponting b A Symonds', 63, 69, '91.3'),
(323, 314, '2004-01-14', 'Zimbabwe', 'Bellerive Oval', 'b S M Ervine', 44, 59, '74.58'),
(324, 315, '2004-01-18', 'Australia', 'Brisbane Cricket Ground', 'c & b A Symonds', 86, 95, '90.53'),
(325, 316, '2004-02-01', 'Australia', 'WACA Ground', 'c M L Hayden b B Lee', 5, 6, '83.33'),
(326, 317, '2004-02-03', 'Zimbabwe', 'WACA Ground', 'c †T Taibu b H H Streak', 3, 8, '37.5'),
(327, 318, '2004-02-06', 'Australia', 'Melbourne Cricket Ground', 'b B Lee', 8, 22, '36.36'),
(328, 319, '2004-02-08', 'Australia', 'Sydney Cricket Ground', 'c B Lee b J N Gillespie', 27, 40, '67.5'),
(329, 320, '2004-03-13', 'Pakistan', 'National Stadium (Karachi)', 'c NavedulHasan b Shoaib Akhtar', 28, 35, '80.0'),
(330, 321, '2004-03-16', 'Pakistan', 'Rawalpindi Cricket Stadium', 'c Abdul Razzaq b Shoaib Malik', 141, 135, '104.44'),
(331, 322, '2004-03-19', 'Pakistan', 'Arbab Niaz Stadium', 'c †Moin Khan b Shabbir Ahmed', 0, 5, '0.0'),
(332, 323, '2004-03-21', 'Pakistan', 'Gaddafi Stadium', 'c †Moin Khan b Shoaib Akhtar', 7, 13, '53.85'),
(333, 324, '2004-03-24', 'Pakistan', 'Gaddafi Stadium', 'c †Moin Khan b Mohammad Sami', 37, 48, '77.08'),
(334, 325, '2004-07-16', 'United Arab Emirates', 'Rangiri Dambulla International Stadium', 'c Fahad Usman b Asim Saeed', 18, 25, '72.0'),
(335, 326, '2004-07-18', 'Sri Lanka', 'Rangiri Dambulla International Stadium', 'c W S Jayantha b D N T Zoysa', 11, 13, '84.62'),
(336, 327, '2004-07-21', 'Bangladesh', 'Sinhalese Sports Club Ground', 'not out', 82, 126, '65.08'),
(337, 328, '2004-07-24', 'Pakistan', 'R Premadasa Stadium', 'c Imran Nazir b Shoaib Malik', 78, 103, '75.73'),
(338, 329, '2004-07-27', 'Sri Lanka', 'R Premadasa Stadium', 'lbw b D N T Zoysa', 18, 21, '85.71'),
(339, 330, '2004-08-01', 'Sri Lanka', 'R Premadasa Stadium', 'b T M Dilshan', 74, 100, '74.0'),
(340, 331, '2004-11-13', 'Pakistan', 'Eden Gardens', 'run out', 16, 17, '94.12'),
(341, 332, '2004-12-23', 'Bangladesh', 'MA Aziz Stadium', 'c †Khaled Mashud b Nazmul Hossain', 19, 32, '59.38'),
(342, 333, '2004-12-27', 'Bangladesh', 'Bangabandhu National Stadium', 'c †Khaled Mashud b Khaled Mahmud', 47, 42, '111.9'),
(343, 334, '2005-04-02', 'Pakistan', 'Nehru Stadium (Kochi)', 'c Mohammad Yousuf b NavedulHasan', 4, 4, '100.0'),
(344, 335, '2005-04-05', 'Pakistan', 'Dr YS Rajasekhara Reddy Cricket Stadium', 'run out', 2, 8, '25.0'),
(345, 336, '2005-04-09', 'Pakistan', 'Keenan Stadium', 'c Younis Khan b Mohammad Sami', 6, 11, '54.55'),
(346, 337, '2005-04-12', 'Pakistan', 'Narendra Modi Stadium', 'b Shoaib Malik', 123, 130, '94.62'),
(347, 338, '2005-04-15', 'Pakistan', 'Green Park', 'c †Kamran Akmal b NavedulHasan', 1, 10, '10.0'),
(348, 339, '2005-04-17', 'Pakistan', 'Arun Jaitley Stadium', 'b Iftikhar Anjum', 9, 15, '60.0'),
(349, 340, '2005-10-25', 'Sri Lanka', 'Vidarbha Cricket Association Ground', 'c †K C Sangakkara b M F Maharoof', 93, 96, '96.88'),
(350, 341, '2005-10-28', 'Sri Lanka', 'Punjab Cricket Association IS Bindra Stadium', 'not out', 67, 69, '97.1'),
(351, 342, '2005-10-31', 'Sri Lanka', 'Sawai Mansingh Stadium', 'c †K C Sangakkara b W P U J C Vaas', 2, 3, '66.67'),
(352, 343, '2005-11-03', 'Sri Lanka', 'Nehru Stadium (Pune)', 'b W P U J C Vaas', 11, 19, '57.89'),
(353, 344, '2005-11-09', 'Sri Lanka', 'Madhavrao Scindia Cricket Ground', 'c M F Maharoof b C R D Fernando', 19, 30, '63.33'),
(354, 345, '2005-11-12', 'Sri Lanka', 'Reliance Stadium', 'c & b D N T Zoysa', 39, 48, '81.25'),
(355, 346, '2005-11-16', 'South Africa', 'Rajiv Gandhi International Stadium', 'c †M V Boucher b S M Pollock', 2, 9, '22.22'),
(356, 347, '2005-11-19', 'South Africa', 'M Chinnaswamy Stadium', 'c sub b S M Pollock', 2, 22, '9.09'),
(357, 348, '2005-11-25', 'South Africa', 'Eden Gardens', 'c †M V Boucher b S M Pollock', 2, 15, '13.33'),
(358, 349, '2005-11-28', 'South Africa', 'Wankhede Stadium', 'c A G Prince b A Nel', 30, 44, '68.18'),
(359, 350, '2006-02-06', 'Pakistan', 'Arbab Niaz Stadium', 'lbw b Arshad Khan', 100, 113, '88.5'),
(360, 351, '2006-02-11', 'Pakistan', 'Rawalpindi Cricket Stadium', 'c †Kamran Akmal b Abdul Razzaq', 42, 43, '97.67'),
(361, 352, '2006-02-13', 'Pakistan', 'Gaddafi Stadium', 'c sub b Abdul Razzaq', 95, 104, '91.35'),
(362, 353, '2006-02-16', 'Pakistan', 'Multan Cricket Stadium', 'c †Kamran Akmal b Mohammad Sami', 0, 3, '0.0'),
(363, 354, '2006-08-18', 'Sri Lanka', 'Sinhalese Sports Club Ground', 'not out', 2, 3, '66.67'),
(364, 355, '2006-09-14', 'West Indies', 'Kinrara Academy Oval', 'not out', 141, 148, '95.27'),
(365, 356, '2006-09-16', 'Australia', 'Kinrara Academy Oval', 'c †B J Haddin b M G Johnson', 12, 17, '70.59'),
(366, 357, '2006-09-20', 'West Indies', 'Kinrara Academy Oval', 'run out', 65, 102, '63.73'),
(367, 358, '2006-09-22', 'Australia', 'Kinrara Academy Oval', 'c M E K Hussey b B Lee', 4, 10, '40.0'),
(368, 359, '2006-10-15', 'England', 'Sawai Mansingh Stadium', 'lbw b S J Harmison', 35, 41, '85.37'),
(369, 360, '2006-10-26', 'West Indies', 'Narendra Modi Stadium', 'b I D R Bradshaw', 29, 45, '64.44'),
(370, 361, '2006-10-29', 'Australia', 'Punjab Cricket Association IS Bindra Stadium', 'c †A C Gilchrist b G D McGrath', 10, 26, '38.46'),
(371, 362, '2006-11-22', 'South Africa', 'Kingsmead', 'b A Nel', 35, 51, '68.63'),
(372, 363, '2006-11-26', 'South Africa', 'Newlands', 'c L L Bosman b S M Pollock', 2, 9, '22.22'),
(373, 364, '2006-11-29', 'South Africa', 'St George''s Park', 'c †M V Boucher b S M Pollock', 1, 3, '33.33'),
(374, 365, '2006-12-03', 'South Africa', 'SuperSport Park', 'c A B de Villiers b J M Kemp', 55, 97, '56.7'),
(375, 366, '2007-01-21', 'West Indies', 'Vidarbha Cricket Association Ground', 'lbw b C H Gayle', 31, 38, '81.58'),
(376, 367, '2007-01-24', 'West Indies', 'Barabati Stadium', 'c D S Smith b D B Powell', 0, 6, '0.0'),
(377, 368, '2007-01-27', 'West Indies', 'MA Chidambaram Stadium', 'c R S Morton b D J Bravo', 60, 66, '90.91'),
(378, 369, '2007-01-31', 'West Indies', 'Reliance Stadium', 'not out', 100, 76, '131.58'),
(379, NULL, '2007-02-08', 'Sri Lanka', 'Eden Gardens', 'did not bat', NULL, NULL, NULL),
(380, 370, '2007-02-11', 'Sri Lanka', 'Madhavrao Scindia Cricket Ground', 'st K C Sangakkara b C M Bandara', 54, 61, '88.52'),
(381, 371, '2007-02-14', 'Sri Lanka', 'Nehru Stadium (Margao)', 'b K M D N Kulasekara', 1, 8, '12.5'),
(382, 372, '2007-03-17', 'Bangladesh', 'Queen''s Park Oval', 'c †Mushfiqur Rahim b Abdur Razzak', 7, 26, '26.92'),
(383, 373, '2007-03-19', 'Bermuda', 'Queen''s Park Oval', 'not out', 57, 29, '196.55'),
(384, 374, '2007-03-23', 'Sri Lanka', 'Queen''s Park Oval', 'b C R D Fernando', 0, 3, '0.0'),
(385, 375, '2007-06-23', 'Ireland', 'Civil Service Cricket Club', 'b R K Whelan', 4, 3, '133.33'),
(386, 376, '2007-06-26', 'South Africa', 'Civil Service Cricket Club', 'run out', 99, 143, '69.23'),
(387, 377, '2007-06-29', 'South Africa', 'Civil Service Cricket Club', 'b T Tshabalala', 93, 106, '87.74'),
(388, 378, '2007-07-01', 'South Africa', 'Civil Service Cricket Club', 'c †M V Boucher b M Ntini', 8, 8, '100.0'),
(389, 379, '2007-08-21', 'England', 'The Rose Bowl', 'c R S Bopara b J M Anderson', 17, 33, '51.52'),
(390, 380, '2007-08-24', 'England', 'County Ground (Bristol)', 'c †M J Prior b A Flintoff', 99, 112, '88.39'),
(391, 381, '2007-08-27', 'England', 'Edgbaston', 'c P D Collingwood b J M Anderson', 8, 19, '42.11'),
(392, 382, '2007-08-30', 'England', 'Old Trafford', 'c A Flintoff b K P Pietersen', 55, 86, '63.95'),
(393, 383, '2007-09-02', 'England', 'Headingley', 'c †M J Prior b J Lewis', 71, 59, '120.34'),
(394, 384, '2007-09-05', 'England', 'Kennington Oval', 'c P D Collingwood b M S Panesar', 94, 81, '116.05'),
(395, 385, '2007-09-08', 'England', 'Lord''s', 'c †M J Prior b A Flintoff', 30, 46, '65.22'),
(396, 386, '2007-09-29', 'Australia', 'M Chinnaswamy Stadium', 'lbw b M G Johnson', 0, 6, '0.0'),
(397, 387, '2007-10-02', 'Australia', 'Nehru Stadium (Kochi)', 'c A Symonds b S R Clark', 16, 25, '64.0'),
(398, 388, '2007-10-05', 'Australia', 'Rajiv Gandhi International Stadium', 'b G B Hogg', 43, 71, '60.56'),
(399, 389, '2007-10-08', 'Australia', 'Sector 16 Stadium', 'run out', 79, 119, '66.39'),
(400, 390, '2007-10-11', 'Australia', 'Reliance Stadium', 'c †A C Gilchrist b B Lee', 47, 73, '64.38'),
(401, 391, '2007-10-14', 'Australia', 'Vidarbha Cricket Association Ground', 'st A C Gilchrist b J R Hopes', 72, 72, '100.0'),
(402, 392, '2007-10-17', 'Australia', 'Wankhede Stadium', 'b B Lee', 21, 36, '58.33'),
(403, 393, '2007-11-05', 'Pakistan', 'Nehru Stadium (Guwahati)', 'lbw b Shoaib Akhtar', 4, 7, '57.14'),
(404, 394, '2007-11-08', 'Pakistan', 'Punjab Cricket Association IS Bindra Stadium', 'c †Kamran Akmal b Umar Gul', 99, 91, '108.79'),
(405, 395, '2007-11-11', 'Pakistan', 'Green Park', 'c †Kamran Akmal b Sohail Tanvir', 29, 27, '107.41'),
(406, 396, '2007-11-15', 'Pakistan', 'Captain Roop Singh Stadium', 'b Umar Gul', 97, 102, '95.1'),
(407, 397, '2007-11-18', 'Pakistan', 'Sawai Mansingh Stadium', 'c MisbahulHaq b Sohail Tanvir', 30, 27, '111.11'),
(408, 398, '2008-02-03', 'Australia', 'Brisbane Cricket Ground', 'hit wicket b B Lee', 10, 17, '58.82'),
(409, 399, '2008-02-04', 'Sri Lanka', 'Brisbane Cricket Ground', 'b S L Malinga', 35, 52, '67.31'),
(410, 400, '2008-02-10', 'Australia', 'Melbourne Cricket Ground', 'c B Lee b M G Johnson', 44, 54, '81.48'),
(411, 401, '2008-02-12', 'Sri Lanka', 'Manuka Oval', 'c K M D N Kulasekara b M F Maharoof', 32, 30, '106.67'),
(412, 402, '2008-02-17', 'Australia', 'Adelaide Oval', 'lbw b N W Bracken', 5, 15, '33.33'),
(413, 403, '2008-02-19', 'Sri Lanka', 'Adelaide Oval', 'b S L Malinga', 0, 2, '0.0'),
(414, 404, '2008-02-24', 'Australia', 'Sydney Cricket Ground', 'lbw b B Lee', 2, 3, '66.67'),
(415, 405, '2008-02-26', 'Sri Lanka', 'Bellerive Oval', 'c L P C Silva b M Muralitharan', 63, 54, '116.67'),
(416, 406, '2008-03-02', 'Australia', 'Sydney Cricket Ground', 'not out', 117, 120, '97.5'),
(417, 407, '2008-03-04', 'Australia', 'Brisbane Cricket Ground', 'c R T Ponting b M J Clarke', 91, 121, '75.21'),
(418, 408, '2008-11-23', 'England', 'M Chinnaswamy Stadium', 'b S C J Broad', 11, 21, '52.38'),
(419, 409, '2008-11-26', 'England', 'Barabati Stadium', 'b S J Harmison', 50, 57, '87.72'),
(420, 410, '2009-01-28', 'Sri Lanka', 'Rangiri Dambulla International Stadium', 'lbw b T Thushara', 5, 16, '31.25'),
(421, 411, '2009-01-31', 'Sri Lanka', 'R Premadasa Stadium', 'lbw b K M D N Kulasekara', 6, 9, '66.67'),
(422, 412, '2009-02-03', 'Sri Lanka', 'R Premadasa Stadium', 'lbw b C R D Fernando', 7, 12, '58.33'),
(423, 413, '2009-03-03', 'New Zealand', 'McLean Park', 'c †B B McCullum b I G Butler', 20, 23, '86.96'),
(424, 414, '2009-03-06', 'New Zealand', 'Westpac Stadium', 'lbw b D L Vettori', 61, 69, '88.41'),
(425, 415, '2009-03-08', 'New Zealand', 'AMI Stadium', 'retired hurt', 163, 133, '122.56'),
(426, 416, '2009-09-11', 'New Zealand', 'R Premadasa Stadium', 'c M J Guptill b D L Vettori', 46, 55, '83.64'),
(427, 417, '2009-09-12', 'Sri Lanka', 'R Premadasa Stadium', 'c B A W Mendis b K M D N Kulasekara', 27, 33, '81.82'),
(428, 418, '2009-09-14', 'Sri Lanka', 'R Premadasa Stadium', 'lbw b B A W Mendis', 138, 133, '103.76'),
(429, 419, '2009-09-26', 'Pakistan', 'SuperSport Park', 'c †Kamran Akmal b Mohammad Amir', 8, 14, '57.14'),
(430, NULL, '2009-09-28', 'Australia', 'SuperSport Park', 'did not bat', NULL, NULL, NULL),
(431, 420, '2009-10-25', 'Australia', 'Reliance Stadium', 'c R T Ponting b S R Watson', 14, 29, '48.28'),
(432, 421, '2009-10-28', 'Australia', 'Vidarbha Cricket Association Stadium', 'c C L White b P M Siddle', 4, 8, '50.0'),
(433, 422, '2009-10-31', 'Australia', 'Arun Jaitley Stadium', 'run out', 32, 47, '68.09'),
(434, 423, '2009-11-02', 'Australia', 'Punjab Cricket Association IS Bindra Stadium', 'lbw b N M Hauritz', 40, 68, '58.82'),
(435, 424, '2009-11-05', 'Australia', 'Rajiv Gandhi International Stadium', 'c N M Hauritz b C J McKay', 175, 141, '124.11'),
(436, 425, '2009-11-08', 'Australia', 'Nehru Stadium (Guwahati)', 'c & b D E Bollinger', 10, 17, '58.82'),
(437, 426, '2009-12-15', 'Sri Lanka', 'Madhavrao Scindia Cricket Ground', 'b C R D Fernando', 69, 63, '109.52'),
(438, 427, '2009-12-18', 'Sri Lanka', 'Vidarbha Cricket Association Stadium', 'st K C Sangakkara b B A W Mendis', 43, 52, '82.69'),
(439, 428, '2009-12-21', 'Sri Lanka', 'Barabati Stadium', 'not out', 96, 104, '92.31'),
(440, 429, '2009-12-24', 'Sri Lanka', 'Eden Gardens', 'c S Randiv b R A S Lakmal', 8, 8, '100.0'),
(441, 430, '2010-02-21', 'South Africa', 'Sawai Mansingh Stadium', 'run out', 4, 5, '80.0'),
(442, 431, '2010-02-24', 'South Africa', 'Captain Roop Singh Stadium', 'not out', 200, 147, '136.05'),
(443, 432, '2011-01-12', 'South Africa', 'Kingsmead', 'c D W Steyn b L L Tsotsobe', 7, 11, '63.64'),
(444, 433, '2011-01-15', 'South Africa', 'Wanderers Stadium', 'b J Botha', 24, 44, '54.55'),
(445, 434, '2011-02-19', 'Bangladesh', 'Shere Bangla National Stadium', 'run out', 28, 29, '96.55'),
(446, 435, '2011-02-27', 'England', 'M Chinnaswamy Stadium', 'c M H Yardy b J M Anderson', 120, 115, '104.35'),
(447, 436, '2011-03-06', 'Ireland', 'M Chinnaswamy Stadium', 'lbw b G H Dockrell', 38, 56, '67.86'),
(448, 437, '2011-03-09', 'Netherlands', 'Arun Jaitley Stadium', 'c B P Kruger b P M Seelaar', 27, 22, '122.73'),
(449, 438, '2011-03-12', 'South Africa', 'Vidarbha Cricket Association Stadium', 'c JP Duminy b M Morkel', 111, 101, '109.9'),
(450, 439, '2011-03-20', 'West Indies', 'MA Chidambaram Stadium', 'c †D C Thomas b R Rampaul', 2, 4, '50.0'),
(451, 440, '2011-03-24', 'Australia', 'Narendra Modi Stadium', 'c †B J Haddin b S W Tait', 53, 68, '77.94'),
(452, 441, '2011-03-30', 'Pakistan', 'Punjab Cricket Association IS Bindra Stadium', 'c Shahid Afridi b Saeed Ajmal', 85, 115, '73.91'),
(453, 442, '2011-04-02', 'Sri Lanka', 'Wankhede Stadium', 'c †K C Sangakkara b S L Malinga', 18, 14, '128.57'),
(454, 443, '2012-02-05', 'Australia', 'Melbourne Cricket Ground', 'c R T Ponting b M A Starc', 2, 6, '33.33'),
(455, 444, '2012-02-08', 'Sri Lanka', 'WACA Ground', 'b A D Mathews', 48, 63, '76.19'),
(456, 445, '2012-02-14', 'Sri Lanka', 'Adelaide Oval', 'c †K C Sangakkara b K M D N Kulasekara', 15, 24, '62.5'),
(457, 446, '2012-02-19', 'Australia', 'Brisbane Cricket Ground', 'c X J Doherty b B W Hilfenhaus', 3, 12, '25.0'),
(458, 447, '2012-02-21', 'Sri Lanka', 'Brisbane Cricket Ground', 'b K M D N Kulasekara', 22, 23, '95.65'),
(459, 448, '2012-02-26', 'Australia', 'Sydney Cricket Ground', 'run out', 14, 15, '93.33'),
(460, 449, '2012-02-28', 'Sri Lanka', 'Bellerive Oval', 'lbw b S L Malinga', 39, 30, '130.0'),
(461, 450, '2012-03-13', 'Sri Lanka', 'Shere Bangla National Stadium', 'c D P M D Jayawardene b R A S Lakmal', 6, 19, '31.58'),
(462, 451, '2012-03-16', 'Bangladesh', 'Shere Bangla National Stadium', 'c †Mushfiqur Rahim b Mashrafe Mortaza', 114, 147, '77.55'),
(463, 452, '2012-03-18', 'Pakistan', 'Shere Bangla National Stadium', 'c Younis Khan b Saeed Ajmal', 52, 48, '108.33');

SELECT * FROM sachin_batting_scores limit 10;

Match,Innings,match_date,Versus,Ground,How_Dismissed,Runs,Balls_faced,strike_rate
1,1,1989-12-18,Pakistan,Jinnah Stadium (Gujwranwala),c Wasim Akram b Waqar Younis,0,2,0.0
2,2,1990-03-01,New Zealand,Carisbrook,c & b S A Thomson,0,2,0.0
3,3,1990-03-06,New Zealand,Basin Reserve,c †I D S Smith b S A Thomson,36,39,92.31
4,4,1990-04-25,Sri Lanka,Sharjah Cricket Stadium,run out,10,12,83.33
5,5,1990-04-27,Pakistan,Sharjah Cricket Stadium,c Saeed Anwar b Imran Khan,20,25,80.0
6,6,1990-07-18,England,Headingley,b D E Malcolm,19,35,54.29
7,7,1990-07-20,England,Trent Bridge,b A R C Fraser,31,26,119.23
8,8,1990-12-01,Sri Lanka,Vidarbha Cricket Association Ground,b R J Ratnayake,36,22,163.64
9,9,1990-12-05,Sri Lanka,Nehru Stadium (Pune),b G F Labrooy,53,41,129.27
10,10,1990-12-08,Sri Lanka,Nehru Stadium (Margao),c & b S D Anurasiri,30,29,103.45


In [0]:
%sql
with cte1 as (
select
    Match,
    Innings,
    sum(Runs)over(order by Match rows between unbounded preceding and current row) as running_total_runs
from sachin_batting_scores
),
cte2 as (
    select 1 as milestone_number, 1000 as milestone_runs
    union all
    select 2 as milestone_number, 5000 as milestone_runs
    union all
    select 3 as milestone_number, 10000 as milestone_runs
    union all
    select 4 as milestone_number, 15000 as milestone_runs
)
select
    milestone_number,
    milestone_runs,
    min(running_total_runs) as player_runs,
    min(match) as milestone_match,
    min(innings) as milestone_innings
from cte1
inner join cte2
on cte1.running_total_runs >= cte2.milestone_runs
group by milestone_number, milestone_runs
order by milestone_number;

milestone_number,milestone_runs,player_runs,milestone_match,milestone_innings
1,1000,1075,36,34
2,5000,5021,141,138
3,10000,10105,266,259
4,15000,15043,387,377
